# HADES Hasar Tespiti - Google Colab Eğitim Notebook'u

Bu notebook, lokal GPU'su olmayan kullanıcılar için **Google Colab** üzerinde HADES modelini eğitmek amacıyla hazırlanmıştır.

## Kullanım Adımları

1. Bu notebook'u Google Colab'da açın
2. `Runtime > Change runtime type > T4 GPU` seçin
3. Hücreleri sırayla çalıştırın
4. Eğitilen `best.pt` dosyasını indirip projenizin `runs/train/hades_egitim/weights/` klasörüne koyun

---

### 1. GPU Kontrolü ve Ultralytics Kurulumu


In [ ]:
#@title GPU Kontrolü
import torch
print(f"CUDA mevcut: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("GPU bulunamadi! Runtime > Change runtime type > T4 GPU secin.")


In [ ]:
#@title Ultralytics Kurulumu
!pip install ultralytics -q
from ultralytics import YOLO, RTDETR
print("Ultralytics kuruldu.")



---

### 2. Google Drive Bağlama ve Veri Seti Yükleme


In [ ]:
#@title Google Drive Bağlama
from google.colab import drive
drive.mount('/content/drive')



**Veri setinizi Drive'a yükleyin:**
1. Google Drive'ınızda `hades_dataset` adında bir klasör oluşturun
2. İçine aşağıdaki yapıyı kurun:


   hades_dataset/
   ├── images/
   │   ├── train/    (eğitim görselleri)
   │   └── val/      (doğrulama görselleri)
   └── labels/
       ├── train/    (eğitim etiketleri .txt)
       └── val/      (doğrulama etiketleri .txt)


In [ ]:
#@title Dosya Yollarını Ayarla
import os
from pathlib import Path

DRIVE_YOLU = Path("/content/drive/MyDrive/hades_dataset")

print(f"Train gorsel: {len(list((DRIVE_YOLU / 'images' / 'train').glob('*')))} adet")
print(f"Val gorsel  : {len(list((DRIVE_YOLU / 'images' / 'val').glob('*')))} adet")
print(f"Train etiket: {len(list((DRIVE_YOLU / 'labels' / 'train').glob('*.txt')))} adet")
print(f"Val etiket  : {len(list((DRIVE_YOLU / 'labels' / 'val').glob('*.txt')))} adet")



---

### 3. Model Seçimi ve Konfigürasyon


In [ ]:
#@title Model Seçimi (YOLO veya RT-DETR)
MODEL_TURU = "yolo"  #@param ["yolo", "rtdetr"]
MODEL_BOYUT = "x"    #@param ["n", "s", "m", "l", "x"] {allow-input: true}

if MODEL_TURU == "yolo":
    MODEL_AGIRLIK = f"yolo12{MODEL_BOYUT}.pt"
else:
    MODEL_AGIRLIK = f"rtdetr-{MODEL_BOYUT}.pt"

print(f"Secilen model: {MODEL_AGIRLIK}")


In [ ]:
#@title dataset.yaml Oluştur
import yaml

dataset_yaml = {
    "path": str(DRIVE_YOLU),
    "train": "images/train",
    "val": "images/val",
    "nc": 7,
    "names": {
        0: "Cizik",
        1: "Gocuk",
        2: "Cam Kirigi",
        3: "Pas",
        4: "Kus Pisligi",
        5: "Far Kirigi",
        6: "Patlak Lastik"
    }
}

yaml_yolu = DRIVE_YOLU / "dataset.yaml"
with open(yaml_yolu, "w", encoding="utf-8") as f:
    yaml.dump(dataset_yaml, f, sort_keys=False, allow_unicode=True)

print(f"dataset.yaml olusturuldu: {yaml_yolu}")



---

### 4. Model Eğitimi


In [ ]:
#@title Eğitimi Başlat
EPOCH = 100      #@param {type:"integer"}
BATCH = 16       #@param {type:"integer"}
IMG_SIZE = 640   #@param {type:"integer"}

if MODEL_TURU == "rtdetr":
    model = RTDETR(MODEL_AGIRLIK)
else:
    model = YOLO(MODEL_AGIRLIK)

print(f"Model: {MODEL_AGIRLIK}")
print(f"Epoch: {EPOCH} | Batch: {BATCH} | ImgSize: {IMG_SIZE}")
print("Egitim basliyor...")

sonuclar = model.train(
    data=str(yaml_yolu),
    epochs=EPOCH,
    batch=BATCH,
    imgsz=IMG_SIZE,
    device=0,
    project="/content/runs",
    name="hades_egitim",
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

print("Egitim tamamlandi!")



---

### 5. Sonuçları İndir


In [ ]:
#@title Eğitim Metriklerini Görüntüle
import pandas as pd

sonuc_csv = "/content/runs/hades_egitim/results.csv"
if os.path.exists(sonuc_csv):
    df = pd.read_csv(sonuc_csv)
    print("Son epoch metrikleri:")
    print(df.tail(1).to_string())
else:
    print("results.csv bulunamadi.")


In [ ]:
#@title best.pt'yi İndir
from google.colab import files
import shutil

best_pt = "/content/runs/hades_egitim/weights/best.pt"
if os.path.exists(best_pt):
    shutil.copy(best_pt, "/content/best.pt")
    files.download("/content/best.pt")
    print("best.pt indirildi! Bu dosyayi projenizin runs/train/hades_egitim/weights/ klasorune koyun.")
else:
    print("best.pt bulunamadi.")


In [ ]:
#@title Tüm Eğitim Klasörünü ZIP Olarak İndir
import zipfile
from google.colab import files

zip_yolu = "/content/hades_egitim_sonuc.zip"
with zipfile.ZipFile(zip_yolu, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk("/content/runs/hades_egitim"):
        for file in files_list:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, "/content/runs")
            zipf.write(file_path, arcname)

files.download(zip_yolu)
print("Tam egitim klasoru ZIP olarak indirildi.")
